# FaithTrace — GPU Inference Demo

This notebook demonstrates the NVIDIA inference optimization pipeline:
1. GPU auto-profiling (pynvml)
2. PyTorch DistilBERT failure classifier training
3. ONNX export with dynamic batch axes
4. TensorRT engine conversion (FP32 / FP16 / INT8)
5. Inference benchmarking (p50/p95/p99 latency + throughput)

**Runtime → Change runtime type → T4 GPU** before running.

## 1. Setup

In [ ]:
# Verify GPU is available
!nvidia-smi

In [ ]:
import os

# Clone only if not already present — safe to re-run
if not os.path.exists('/content/FaithTrace/.git'):
    !git clone https://github.com/sakshiasati17/FaithTrace.git /content/FaithTrace

# Always use absolute path — never nests even if cell runs twice
%cd /content/FaithTrace
!echo "Working directory: $(pwd)"

In [ ]:
# Install core dependencies
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers onnx onnxruntime-gpu pynvml numpy onnxscript

In [ ]:
# Add backend to path
import sys
sys.path.insert(0, 'backend')

## 2. GPU Auto-Profiling

In [ ]:
from app.optimization.gpu_profiler import GPUAutoProfiler

profiler = GPUAutoProfiler()
profile = profiler.profile()

print(f"GPU: {profile.name}")
print(f"VRAM Total: {profile.vram_total_gb} GB")
print(f"VRAM Available: {profile.vram_available_gb} GB")
print(f"Compute Capability: {profile.compute_capability}")
print(f"Recommended Precision: {profile.recommended_precision}")
print(f"Max Batch Size: {profile.max_batch_size}")
print(f"\nReasoning: {profile.reasoning}")

## 3. PyTorch DistilBERT Failure Classifier

In [ ]:
import torch
from app.models.failure_classifier import FailureClassifier
from app.models.failure_classifier.dataset import LABEL_NAMES

model = FailureClassifier(num_classes=6)
model.eval()
model.cuda()

print(f"Model: DistilBERT encoder + RAGAS score fusion + 6-class head")
print(f"Classes: {LABEL_NAMES}")
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Frozen params: {sum(p.numel() for p in model.parameters() if not p.requires_grad):,}")
print(f"Device: {next(model.parameters()).device}")

In [ ]:
# Test forward pass with dummy data
from transformers import DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

sample_queries = [
    "What is the procurement threshold for emergency purchases?",
    "Show the Q1 revenue breakdown by region from the financial report",
    "What changed in version 3 of the vendor onboarding policy?",
    "What are the safety inspection intervals for heavy equipment?",
]

tokens = tokenizer(sample_queries, padding=True, truncation=True, max_length=128, return_tensors='pt')
input_ids = tokens['input_ids'].cuda()
attention_mask = tokens['attention_mask'].cuda()

# Dummy RAGAS scores (faithfulness, context_recall, context_precision, answer_relevance, answer_correctness)
ragas_scores = torch.tensor([
    [0.85, 0.72, 0.90, 0.88, 0.76],
    [0.20, 0.15, 0.30, 0.45, 0.22],  # likely table_retrieval_miss
    [0.40, 0.60, 0.55, 0.70, 0.35],  # likely context_insufficient
    [0.95, 0.88, 0.92, 0.90, 0.91],  # likely no_failure
], dtype=torch.float32).cuda()

with torch.no_grad():
    logits = model(input_ids, attention_mask, ragas_scores)
    preds = torch.argmax(logits, dim=1)

print("\nPredictions (untrained model — random):")
for q, p in zip(sample_queries, preds):
    print(f"  {q[:60]}...  →  {LABEL_NAMES[p.item()]}")

print(f"\nLogits shape: {logits.shape}")
print(f"Softmax probabilities:\n{torch.softmax(logits, dim=1).cpu().numpy().round(3)}")

## 4. ONNX Export

In [ ]:
import os
os.makedirs('models', exist_ok=True)

from app.optimization.onnx_export import export_classifier_to_onnx

model.cpu()
onnx_path = export_classifier_to_onnx(model, output_path='models/failure_classifier.onnx')
print(f"\nONNX model exported to: {onnx_path}")
print(f"File size: {os.path.getsize(onnx_path) / 1024 / 1024:.1f} MB")

In [ ]:
# Validate ONNX model
import onnx

onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
print("ONNX model validation: PASSED")
print(f"IR version: {onnx_model.ir_version}")
print(f"Opset: {onnx_model.opset_import[0].version}")
print(f"\nInputs:")
for inp in onnx_model.graph.input:
    print(f"  {inp.name}: {[d.dim_param or d.dim_value for d in inp.type.tensor_type.shape.dim]}")
print(f"\nOutputs:")
for out in onnx_model.graph.output:
    print(f"  {out.name}: {[d.dim_param or d.dim_value for d in out.type.tensor_type.shape.dim]}")

## 5. ONNX Runtime Inference Comparison

In [ ]:
import onnxruntime as ort
import numpy as np
import time

# ONNX was traced with batch_size=1. DistilBERT's internal attention reshapes
# get baked in at trace time — dynamic_axes only covers outer inputs, not
# internal ops. Running batch_size=1 avoids the reshape mismatch.
# For throughput, both backends are benchmarked at batch_size=1 (per-query).
tokenizer_onnx = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
tokens_single = tokenizer_onnx(
    [sample_queries[0]], padding='max_length', truncation=True, max_length=512, return_tensors='pt'
)
input_ids_np   = tokens_single['input_ids'].numpy()       # (1, 512)
attn_mask_np   = tokens_single['attention_mask'].numpy()  # (1, 512)
ragas_np       = ragas_scores[:1].cpu().numpy()            # (1, 5)

# Setup ONNX Runtime with GPU
providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
session = ort.InferenceSession(onnx_path, providers=providers)
print(f"ONNX Runtime provider: {session.get_providers()[0]}")

def run_ort():
    return session.run(None, {
        'input_ids':      input_ids_np,
        'attention_mask': attn_mask_np,
        'ragas_scores':   ragas_np,
    })

# Prepare batch_size=1 PyTorch inputs for fair comparison
input_ids_pt = tokens_single['input_ids'].cuda()
attn_mask_pt = tokens_single['attention_mask'].cuda()
ragas_pt     = ragas_scores[:1]

def run_pt():
    with torch.no_grad():
        return model(input_ids_pt, attn_mask_pt, ragas_pt)

# Warmup
model.cuda().eval()
for _ in range(10):
    run_ort()
for _ in range(10):
    run_pt()
torch.cuda.synchronize()

# Benchmark
n_runs = 100
ort_times, pt_times = [], []

for _ in range(n_runs):
    start = time.perf_counter()
    run_ort()
    ort_times.append((time.perf_counter() - start) * 1000)

for _ in range(n_runs):
    torch.cuda.synchronize()
    start = time.perf_counter()
    run_pt()
    torch.cuda.synchronize()
    pt_times.append((time.perf_counter() - start) * 1000)

ort_times = sorted(ort_times)
pt_times  = sorted(pt_times)

print(f"\n{'='*60}")
print(f"  INFERENCE BENCHMARK (batch_size=1, {n_runs} runs)")
print(f"{'='*60}")
print(f"{'Metric':<20} {'PyTorch':>12} {'ONNX Runtime':>14} {'Speedup':>10}")
print(f"{'-'*60}")
for label, idx in [('p50 (ms)', n_runs//2), ('p95 (ms)', int(n_runs*0.95)), ('p99 (ms)', int(n_runs*0.99))]:
    pt_val  = pt_times[idx]
    ort_val = ort_times[idx]
    speedup = pt_val / ort_val if ort_val > 0 else 0
    print(f"{label:<20} {pt_val:>10.2f}ms {ort_val:>12.2f}ms {speedup:>9.1f}x")

pt_qps  = 1000 / np.mean(pt_times)
ort_qps = 1000 / np.mean(ort_times)
print(f"{'Throughput':<20} {pt_qps:>10.0f}qps {ort_qps:>12.0f}qps {ort_qps/pt_qps:>9.1f}x")
print(f"{'='*60}")

## 6. TensorRT Conversion (if available)

In [ ]:
# Try installing TensorRT (may not work on all Colab instances)
!pip install -q tensorrt 2>/dev/null || echo "TensorRT not available in this Colab runtime"

In [ ]:
from app.optimization.tensorrt_convert import TRT_AVAILABLE, convert_all_precisions

if TRT_AVAILABLE:
    print("TensorRT is available! Converting...")
    os.makedirs('models/tensorrt', exist_ok=True)
    engines = convert_all_precisions(onnx_path, output_dir='models/tensorrt')
    for precision, path in engines.items():
        size_mb = os.path.getsize(path) / 1024 / 1024
        print(f"  {precision}: {path} ({size_mb:.1f} MB)")
else:
    print("TensorRT not available in this runtime.")
    print("This is expected on free Colab — the code handles this gracefully.")
    print("\nIn production with an NVIDIA GPU + TensorRT installed:")
    print("  - FP32 engine: baseline TensorRT performance")
    print("  - FP16 engine: ~2x speedup, minimal accuracy loss")
    print("  - INT8 engine: ~3-4x speedup, requires calibration")

## 7. Focal Loss Demo

In [ ]:
from app.models.failure_classifier.losses import FocalLoss

focal_loss = FocalLoss(num_classes=6, gamma=2.0, label_smoothing=0.1)

# Simulate imbalanced batch: mostly no_failure (class 0), few hallucinations (class 3)
dummy_logits = torch.randn(16, 6).cuda()
dummy_labels = torch.tensor([0,0,0,0,0,0,0,0,0,0,1,1,2,3,4,5]).cuda()  # 10 no_failure, 6 failures

loss = focal_loss(dummy_logits, dummy_labels)
print(f"Focal Loss: {loss.item():.4f}")
print(f"  gamma={focal_loss.gamma} (down-weights easy examples)")
print(f"  label_smoothing={focal_loss.label_smoothing} (prevents overconfidence)")
print(f"  alpha weights per class: {focal_loss.alpha.cpu().numpy().round(3)}")

# Compare with standard CrossEntropy
ce_loss = torch.nn.CrossEntropyLoss()(dummy_logits, dummy_labels)
print(f"\nStandard CrossEntropy: {ce_loss.item():.4f}")
print(f"Focal Loss focuses more on hard/rare examples → better for imbalanced failure classes")

## 8. Summary

In [ ]:
print(f"""{'='*60}
  FaithTrace GPU Inference Pipeline — Summary
{'='*60}

GPU Profile:
  Device:     {profile.name}
  VRAM:       {profile.vram_total_gb} GB total / {profile.vram_available_gb} GB free
  Precision:  {profile.recommended_precision} (recommended)
  Batch Size: {profile.max_batch_size} (max recommended)

Model:
  Architecture:  DistilBERT + RAGAS fusion (773-dim) + 6-class head
  Parameters:    {sum(p.numel() for p in model.parameters()):,} total
  Trainable:     {sum(p.numel() for p in model.parameters() if p.requires_grad):,}
  Loss:          Focal Loss (gamma=2.0, label_smoothing=0.1)

Export:
  ONNX:          {onnx_path} ({os.path.getsize(onnx_path)/1024/1024:.1f} MB)
  Dynamic Batch: batch_size=1 (DistilBERT attention reshapes traced at export time)
  TensorRT:      {'Available' if TRT_AVAILABLE else 'Not available (graceful fallback)'}

Benchmark (batch_size=1, {n_runs} runs):
  PyTorch p50:      {pt_times[n_runs//2]:.2f} ms
  ONNX Runtime p50: {ort_times[n_runs//2]:.2f} ms
  Speedup:          {pt_times[n_runs//2]/ort_times[n_runs//2]:.1f}x

{'='*60}""")